# DrVideo on the office laptop (Qwen API)

This is the office version of the DrVideo notebook. It works exactly like the laptop (Ollama) notebook, block for block. The only difference is **how we reach Qwen**: here we call your Qwen server through its `base_url` and `api_key`.

| Job | Model | Runs on |
|---|---|---|
| Look at frames (captions, details) | Qwen-VL | your Qwen server (API) |
| Reasoning agents (planning, interaction, answer) | Qwen (can be the same Qwen-VL model) | your Qwen server (API) |
| Text → vector (retrieval) | `BAAI/bge-large-en-v1.5` (BGE) | `sentence-transformers` (pip), on CPU |

**The idea of the paper:** a model can't "watch" a long video well, but it can read a long text well. So we turn the video into a text document, one line per frame, and let the LLM reason over that document.

**The 5 steps:**
1. **Video → document**: grab a frame every few seconds and caption each one.
2. **Retrieval**: find the 5 captions most similar to the question.
3. **Augmentation**: look at those 5 frames again and write more detailed notes.
4. **Agent loop**: ask "is this enough to answer?" If not, pick more frames to look at.
5. **Answer**: read the final document and answer.

**Blocks 1–5 differ from the laptop notebook. From Block 6 onwards, the code is identical.**

## Block 1: Install the packages

**What this block does**
- `openai` is the client library for OpenAI-style APIs. Most Qwen servers (Alibaba DashScope, vLLM and others) speak this same API, so this library works with yours: we just point it at your `base_url`.
- `sentence-transformers` runs the BGE embedding model. It installs PyTorch too, so the first install takes a few minutes. The BGE model itself (~1.3 GB) downloads automatically the first time Block 6 runs.
- `opencv-python` reads video frames.
- `numpy` does the math.

The `!` at the start of the line runs it as a terminal command from inside Jupyter. You only need to run this block once.

In [ ]:
!pip install -q openai sentence-transformers opencv-python numpy

## Block 2: Imports

**What each import is for**
- `base64`: turn image bytes into text, which is how images travel inside an API request
- `json`: save and load captions, and read the JSON replies from the model
- `os`: check whether a file exists
- `re`: regular expressions, used to find the JSON inside a reply
- `cv2` (OpenCV): open the video and grab frames
- `numpy`: vector math for retrieval
- `OpenAI`: the client that talks to your Qwen server
- `SentenceTransformer`: load and run the BGE model

In [ ]:
import base64
import json
import os
import re

import cv2
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

## Block 3: Settings

All the knobs are in one place, so you can change them easily.

**Your Qwen server**
- `BASE_URL` and `API_KEY` are the two values you were given. Keep this notebook private, since anyone with the key can use your Qwen account.
- `VL_MODEL` is the exact model name your server uses for Qwen-VL, for example `qwen-vl-max` or `Qwen/Qwen2.5-VL-72B-Instruct`. Ask whoever runs the server if you're not sure.
- `LLM_MODEL` is used by the reasoning agents. A Qwen-VL model can read plain text too, so it can be the same name. If your server also has a text-only Qwen model, you can use that here instead.

**The DrVideo knobs** (same values as the laptop notebook, from the paper)
- `TOP_K = 5`: how many frames retrieval picks. The paper found 5 works best; more frames add noise.
- `NUM_NEW_FRAMES = 3`: how many extra frames the agent may ask for in each round. The paper requires this to be smaller than `TOP_K`.
- `MAX_ROUNDS = 2`: how many times the agent may ask for more frames. The paper found 2 is best.

**`FPS` and your 120k context**
- Every agent prompt contains the whole document, about 75 tokens per frame.
- Your Qwen reads up to 120k tokens, which is room for about 1,500 frames.
- At `FPS = 0.5` (1 frame every 2 seconds), that's videos up to about **45 minutes**.
- For longer videos, set `FPS = 0.2` (1 frame every 5 seconds, as the paper did for hour-long videos). That fits about **2 hours**.

Unlike the laptop, there's no `num_ctx` to set: the context size is decided by the server, not by us.

In [ ]:
BASE_URL = "your-base-url"
API_KEY = "your-api-key"
VL_MODEL = "qwen-vl-max"     # your Qwen-VL model name
LLM_MODEL = VL_MODEL         # model for the agents (can be the same one)

EMBED_MODEL = "BAAI/bge-large-en-v1.5"

TOP_K = 5
NUM_NEW_FRAMES = 3
MAX_ROUNDS = 2
FPS = 0.5                    # use 0.2 for videos longer than ~45 minutes

VIDEO_PATH = "my_video.mp4"
QUESTION = "What is the person doing in this video?"

## Block 4: Talk to Qwen

These two small functions are how every other block talks to the model. They do the same job as the laptop versions, through the API instead of Ollama.

**`client = OpenAI(...)`** creates one connection object that points at your server. Every call goes through it.

**`ask_llm(prompt)`** sends text only. The agents use it.
- `messages` is a conversation: a list of turns, each with a `role` and `content`. We send one user turn, so every call is a fresh, standalone question.
- `temperature=0` means no randomness, so the same input gives the same output (the paper's setting).
- The reply text is at `response.choices[0].message.content`. The API can return several alternative replies ("choices"); we only ask for one, so we take the first.

**`ask_vlm(image, prompt)`** sends an image plus text.
- `image` is the JPEG bytes of a frame (Block 7).
- APIs send everything as text, so we turn the bytes into a **base64** string and wrap it as a *data URL*: `data:image/jpeg;base64,...`. That is how the OpenAI-style API accepts an image inside a message.
- The message `content` is now a **list** of two parts: the image and the text.
- `max_tokens` caps how long the reply can be. It stops captions from running too long.

**Two small safety lines**
- `or ""`: some servers return `None` instead of an empty string, which would crash `.strip()` later.
- `re.sub(r"<think>.*?</think>", ...)`: some Qwen models (Qwen3 and others) write their private reasoning inside `<think>...</think>` before the real answer. This line deletes that part, so only the answer is left. If your model doesn't do this, the line simply changes nothing.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)


def ask_llm(prompt):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    text = response.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def ask_vlm(image, prompt, max_words=300):
    image_url = "data:image/jpeg;base64," + base64.b64encode(image).decode()
    response = client.chat.completions.create(
        model=VL_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": image_url}},
            {"type": "text", "text": prompt},
        ]}],
        temperature=0,
        max_tokens=max_words,
    )
    text = response.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

**Quick check:** make sure Python can reach your Qwen server.

- If you get an **authentication** error, check `API_KEY`.
- If you get a **connection** error, check `BASE_URL`. It usually ends in `/v1`.
- If you get a **model not found** error, check `VL_MODEL`.

In [ ]:
print(ask_llm("Say hello in exactly 5 words."))

## Block 5: Read JSON safely

The agents ask Qwen to reply in JSON, like `{"confidence": "1", "explanation": "..."}`. Unlike the laptop notebook, there's no "JSON mode" switch here. Not every Qwen server supports one, so we stay compatible with all of them. That means Qwen sometimes adds extra text around the JSON, for example:

````
Sure! Here is my answer:
```json
{"confidence": "1", "explanation": "..."}
```
````

**How this function copes**
1. `re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)` finds the part that starts with `{` and ends with `}` (or `[` ... `]`), skipping the chatty text around it. `re.DOTALL` lets `.` match newlines too, so JSON spread over several lines is still found.
2. `json.loads(...)` turns that JSON text into a Python dict or list.
3. If there's no JSON or it's broken, we return `None` instead of crashing, and the calling code handles `None`.

In [ ]:
def read_json(text):
    match = re.search(r"\{.*\}|\[.*\]", text, re.DOTALL)
    if match is None:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

## Block 6: Embeddings with BGE

An **embedding** turns a sentence into a list of numbers (a vector, 1024 numbers for bge-large). Sentences with similar meaning get vectors that point in similar directions.

- `SentenceTransformer(EMBED_MODEL)` downloads the model from Hugging Face the first time, then loads it from your disk on later runs.
- `device="cpu"` runs BGE on the CPU, so it works on any laptop. BGE is small and fast: embedding a few hundred captions on CPU takes only seconds.
- `normalize_embeddings=True` makes every vector length 1. With that, the dot product of two vectors gives their **cosine similarity**: 1 means the same meaning, 0 means unrelated. This is how retrieval finds frames related to the question (paper Eq. 2).

In [ ]:
bge = SentenceTransformer(EMBED_MODEL, device="cpu")


def embed(texts):
    return bge.encode(texts, normalize_embeddings=True)

**Quick check (try it!):** the first two sentences mean similar things, and the third is unrelated. You should see a high score for the first pair and a low one for the second.

In [ ]:
v = embed(["A man is washing dishes", "Someone cleans plates in the sink", "A dog runs in the park"])
print("dishes vs plates:", round(float(v[0] @ v[1]), 3))
print("dishes vs dog:   ", round(float(v[0] @ v[2]), 3))

## Block 7: Read frames from the video

**How it works**
1. Open the video and work out its length: `duration` = total frames ÷ the video's own fps.
2. Jump to second 0, 2, 4, 6… (every `1 / FPS` seconds). `CAP_PROP_POS_MSEC` jumps to a time in milliseconds.
3. Read the frame there and shrink it to 512 px wide. A smaller image costs fewer tokens, so API calls are faster and cheaper.
4. Encode it as JPEG bytes. `ask_vlm` (Block 4) turns these bytes into base64 for the API.

Frame numbers start at **1**, like in the paper, so `frames[0]` is "Frame 1". Some videos don't report their fps, so `or 30` is a safe fallback.

In [ ]:
def read_frames(video_path, fps=FPS):
    video = cv2.VideoCapture(video_path)
    video_fps = video.get(cv2.CAP_PROP_FPS) or 30
    duration = video.get(cv2.CAP_PROP_FRAME_COUNT) / video_fps

    frames = []
    second = 0.0
    while second < duration:
        video.set(cv2.CAP_PROP_POS_MSEC, second * 1000)
        ok, frame = video.read()
        if ok:
            height, width = frame.shape[:2]
            frame = cv2.resize(frame, (512, int(height * 512 / width)))
            _, jpg = cv2.imencode(".jpg", frame)
            frames.append(jpg.tobytes())
        second += 1 / fps

    video.release()
    print(f"Video is {duration:.0f} seconds long, got {len(frames)} frames")
    return frames, duration

**Quick check:** read your video. If you get 0 frames, check `VIDEO_PATH` in Block 3.

In [ ]:
frames, duration = read_frames(VIDEO_PATH)

## Block 8: Turn the video into a document (paper Eq. 1)

Qwen-VL writes a short caption, under 50 words, for **every** frame. That list of captions is the paper's "initial video document".

This is the **slowest step**, since it's one model call per frame. So we save the captions to a JSON file next to the video. The next time you ask a question about the same video, the captions load instantly from that file. To redo the captions, delete the `.captions.json` file.

`max_words=100` (passed on as `max_tokens`) stops the model from writing long captions, which would fill up the context window.

In [ ]:
CAPTION_PROMPT = "Describe this image in no more than 50 words. Focus on people, objects, actions and the scene."


def make_document(video_path, frames):
    save_file = video_path + ".api_captions.json"
    if os.path.exists(save_file):
        with open(save_file) as f:
            return json.load(f)

    captions = []
    for i, frame in enumerate(frames):
        captions.append(ask_vlm(frame, CAPTION_PROMPT, max_words=100))
        print(f"captioned frame {i + 1}/{len(frames)}", end="\r")

    with open(save_file, "w") as f:
        json.dump(captions, f)
    return captions

To show the document to the LLM, we write one line per frame, like `Frame 12: a man is cutting onions`.

`notes` holds the extra details we gather later (Block 10). It's a dict of `frame number -> list of notes`, for example `{12: ["Detailed description: ..."]}`. The notes are added in `[...]` at the end of that frame's line, so the LLM sees them in the right place in time.

`enumerate(captions, start=1)` counts from 1 instead of 0.

In [ ]:
def document_text(captions, notes):
    lines = []
    for number, caption in enumerate(captions, start=1):
        line = f"Frame {number}: {caption}"
        for note in notes.get(number, []):
            line += f" [{note}]"
        lines.append(line)
    return "\n".join(lines)

**Quick check:** caption the video and look at the first few lines. This is the slow part, so watch the counter.

In [ ]:
captions = make_document(VIDEO_PATH, frames)
print(document_text(captions, {})[:1500])

## Block 9: Retrieval (paper Eq. 2)

**Goal:** find the frames most related to the question, without the LLM reading anything yet.

1. Embed all captions, giving one vector per frame.
2. Embed the question. BGE's English models work better when a search question starts with the instruction *"Represent this sentence for searching relevant passages: "*, as their model card recommends. Captions don't get the instruction, only the question.
3. `caption_vectors @ question_vector` computes the dot product of the question with every caption at once, giving one similarity score per frame.
4. `np.argsort(-scores)` sorts from the highest score to the lowest. Take the first `k`.
5. Add 1, because Python counts from 0 but our frame numbers start at 1.

In [ ]:
def retrieve(question, captions, k=TOP_K):
    caption_vectors = embed(captions)
    question_vector = embed(["Represent this sentence for searching relevant passages: " + question])[0]

    scores = caption_vectors @ question_vector
    best = np.argsort(-scores)[:k]
    return sorted(int(i) + 1 for i in best)

**Quick check:** which frames match your question best? Compare them with their captions above.

In [ ]:
print("Top frames:", retrieve(QUESTION, captions))

## Block 10: Augmentation (paper Eq. 3)

A 50-word caption misses details. "A woman looks in the mirror" doesn't tell you what she's wearing. So for important frames we ask Qwen-VL **again** with a more focused prompt and attach its answer as a note.

There are 3 prompt types:
- `"first"`: the paper's prompt for the top-K retrieved frames. If the question doesn't fit the image, it describes the image precisely; otherwise it answers the question.
- `"A"`: a detailed description of the whole image.
- `"B"`: answer the question from this one image (visual question answering).

`requests` is a dict saying which frame gets which type, for example `{12: "A", 40: "B"}`. `notes.setdefault(number, [])` creates an empty list for a frame the first time it gets a note.

In [ ]:
AUGMENT_PROMPTS = {
    "first": "If there are factual errors in the question, provide a precise description of the image; "
             "if not, proceed to answer the question: {question}",
    "A": "Describe this image in detail: people, objects, colors, text, actions and the scene.",
    "B": "Answer this question using the image. If the image cannot answer it, describe what is relevant. "
         "Question: {question}",
}


def augment(frames, notes, requests, question):
    for number, kind in requests.items():
        prompt = AUGMENT_PROMPTS[kind].format(question=question)
        answer = ask_vlm(frames[number - 1], prompt)
        label = "Detailed description" if kind == "A" else "Question-related detail"
        notes.setdefault(number, []).append(f"{label}: {answer}")
        print(f"augmented frame {number} (type {kind})")

## Block 11: Shared intro for the agents

All three agents get the same opening text, which explains what the document is: how long the video is, how many frames it has, and how each line is formatted. The wording follows the prompts in the paper's appendix.

In [ ]:
def video_intro(duration, num_frames):
    return (
        "You are given some language descriptions of a video along with a question about the video.\n"
        f"1. The video is {duration / 60:.1f} minutes long, containing a total of {num_frames} frames.\n"
        "2. Each sentence in these language descriptions represents the text description for a single frame.\n"
        "3. The format of each sentence is frame id: description. The frame id indicates the temporal "
        f"position of the frame, ranging from 1 to {num_frames}.\n"
    )

## Block 12: Planning agent (paper Sec. 3.4)

**Its one job:** read the document and decide whether it's **enough** to answer the question.

- It replies with `confidence` 1 (enough) or 0 (not enough), plus an explanation.
- When the answer is 0, the explanation says **what is missing**, for example "we need to see what the person holds in the kitchen". We save that in `memory` so the next agent knows what to look for.

`{{` and `}}` in an f-string print a literal `{` and `}`. That lets us show the model the JSON format we want. `result.get("confidence", "0")` returns `"0"` if the key is missing, and `or {}` handles a `None` from `read_json`.

In [ ]:
def planning_agent(intro, document, question, memory):
    prompt = f"""{intro}
Here are the descriptions of this video:
{document}

Here is the question: {question}
Here is the memory: {memory}

Your task is to determine whether these descriptions can answer the question accurately, reasonably, and without contradiction.
If yes, the confidence is 1 and give a reasonable explanation.
If no, the confidence is 0 and explain what frame information is missing, focusing on objects, humans, interactions, actions and scenes.
Reply in JSON like this: {{"confidence": "0 or 1", "explanation": "..."}}"""

    result = read_json(ask_llm(prompt)) or {}
    enough = str(result.get("confidence", "0")).strip() == "1"
    return enough, str(result.get("explanation", ""))

## Block 13: Interaction agent (paper Sec. 3.4)

**Its job:** when the document isn't enough, pick **which frames** to look at again, and **what kind** of information to get from each:
- **A**: a detailed description
- **B**: answer the question from that frame

It replies like `{"frames": [{"frame": 12, "type": "A"}]}`.

The model can make mistakes, so we **check every pick** and drop:
- frame numbers outside 1…number of frames
- frames that retrieval already picked (the paper's Eq. 4 says new frames must be different)
- a frame/type pair we already have a note for

`int(pick["frame"])` can fail if the model writes nonsense, so it's wrapped in `try/except` and bad picks are skipped. Finally, we keep at most `NUM_NEW_FRAMES` picks.

In [ ]:
def interaction_agent(intro, document, question, memory, num_frames, retrieved, notes):
    prompt = f"""{intro}
Here are the descriptions of this video:
{document}

Here is the memory: {memory}
To answer the following question: {question}
These descriptions are insufficient and cannot answer this question accurately, reasonably, and without contradiction.
Your task is to determine which frame needs which type of information to answer the question.
The two types of information are:
A: Given an image, get a detailed description of the image.
B: Given an image, get a response to the above question (visual question answering).
Pick at most {NUM_NEW_FRAMES} frames between 1 and {num_frames}. Do not pick these frames: {retrieved}.
Reply in JSON like this: {{"frames": [{{"frame": 12, "type": "A"}}, {{"frame": 40, "type": "B"}}]}}"""

    result = read_json(ask_llm(prompt)) or {}
    picks = result.get("frames", []) if isinstance(result, dict) else result

    requests = {}
    for pick in picks:
        try:
            number = int(pick["frame"])
            kind = str(pick["type"]).upper()
        except (KeyError, ValueError, TypeError):
            continue
        label = "Detailed" if kind == "A" else "Question"
        already_have = any(note.startswith(label) for note in notes.get(number, []))
        if 1 <= number <= num_frames and number not in retrieved and kind in ("A", "B") and not already_have:
            requests[number] = kind
    return dict(list(requests.items())[:NUM_NEW_FRAMES])

## Block 14: Answering agent (paper Sec. 3.5)

The LLM reads the **final** document and answers the question in its own words.

- We ask it to **think step by step** first (chain-of-thought). Writing out its reasoning before answering helps the model get it right, and it lets you see *why* it answered that way.
- It also gives a confidence from 1 to 3.

If the reply isn't valid JSON, we keep the raw text as the answer, so you never lose the result.

In [ ]:
def answer_agent(intro, document, question):
    prompt = f"""{intro}
Here are the descriptions of this video:
{document}

Please answer the following question: {question}

Please think step by step, then give a short, direct answer based only on the descriptions.
The confidence must be 1, 2 or 3 (3 = very sure).
Reply in JSON like this: {{"reasoning": "...", "answer": "...", "confidence": "2"}}"""

    reply = ask_llm(prompt)
    return read_json(reply) or {"answer": reply}

## Block 15: DrVideo, everything together (paper Algorithm 1)

This function just calls the blocks above in the paper's order:

1. Read frames and build the caption document (Blocks 7–8).
2. Retrieve the top-K frames and augment them with the `"first"` prompt (Blocks 9–10).
3. Loop up to `MAX_ROUNDS` times:
   - The **planning agent** checks the document. If it's enough, stop.
   - If not, save its explanation in `memory`. The **interaction agent** picks new frames, and we **augment** them.
   - If the interaction agent has no valid picks, stop early.
4. The **answering agent** answers from the final document.

`memory` is a simple list of strings. It's the paper's "history H", and it lets each step see what was already tried.

In [ ]:
def drvideo(video_path, question):
    frames, duration = read_frames(video_path)
    captions = make_document(video_path, frames)
    intro = video_intro(duration, len(frames))
    notes = {}

    retrieved = retrieve(question, captions)
    print("Retrieved frames:", retrieved)
    augment(frames, notes, {number: "first" for number in retrieved}, question)
    memory = [f"Frames {retrieved} were retrieved as the key frames and given extra details."]

    for round_number in range(1, MAX_ROUNDS + 1):
        document = document_text(captions, notes)
        enough, explanation = planning_agent(intro, document, question, memory)
        print(f"Round {round_number}: enough info = {enough}. {explanation}")
        if enough:
            break

        memory.append(explanation)
        requests = interaction_agent(intro, document, question, memory, len(frames), retrieved, notes)
        print(f"Round {round_number}: looking again at {requests}")
        if not requests:
            break
        augment(frames, notes, requests, question)
        memory.append(f"Frames {list(requests)} were given extra details.")

    final_document = document_text(captions, notes)
    return answer_agent(intro, final_document, question), final_document

## Block 16: Run it!

This uses `VIDEO_PATH` and `QUESTION` from Block 3. Since the captions are already saved, only the retrieval, augmentation and agent steps run now. Try asking different questions about the same video.

In [ ]:
result, final_document = drvideo(VIDEO_PATH, QUESTION)

print("\nAnswer:    ", result.get("answer"))
print("Confidence:", result.get("confidence"))
print("Reasoning: ", result.get("reasoning"))

To see exactly what the LLM read, print the final document. Lines with `[...]` notes are the frames DrVideo looked at more closely. Check whether they're the frames you would have picked.

In [ ]:
print(final_document)